## 1. Install Dependencies

In [3]:
# !pip install gymnasium[box2d]
# !pip install stable-baselines3[extra]

## 2. Imports

In [4]:
import gymnasium as gym
import numpy as np
import os

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import get_linear_fn

def linear_schedule(initial_value: float):
    """Linear schedule from initial_value down to 0."""
    return get_linear_fn(initial_value, 0.0, 1.0)

## 3. Create Environment


In [ ]:
def make_env(render_mode=None):
    def _init():
        env = gym.make("CarRacing-v3", continuous=True, render_mode=render_mode)
        env = Monitor(env)  
        return env
    return _init


def build_env(n_envs=1, render_mode=None, n_stack=4):
    """Build a vectorized + frame-stacked environment."""
    env = DummyVecEnv([make_env(render_mode) for _ in range(n_envs)])
    env = VecTransposeImage(env)      
    env = VecFrameStack(env, n_stack=n_stack)  
    return env


# Training environment
train_env = build_env(n_envs=1)

## 4. Create Model


- `learning_rate`: switched to `linear_schedule(2.5e-4)` — linearly decays to 0, preventing the KL explosion seen in iterations 14-23 of your logs.
- `n_steps=512` with `batch_size=128` — better gradient estimates per update for this env.
- `n_epochs=4` — fewer epochs per rollout reduces overfitting to stale data.
- `ent_coef=0.01` — slightly higher entropy encourages exploration and avoids getting stuck.
- `clip_range=linear_schedule(0.2)` — also decays, tightening the policy update as training progresses.
- `max_grad_norm=0.5` — clips gradients to stabilize training.

In [ ]:
model = PPO(
    policy="CnnPolicy",
    env=train_env,
    
    learning_rate=linear_schedule(2.5e-4),
    
    n_steps=512,         
    batch_size=128,        
    n_epochs=4,            
   
    gamma=0.99,
    gae_lambda=0.95,
  
    clip_range=linear_schedule(0.2),   
    clip_range_vf=None,   
  
    ent_coef=0.01,         
    vf_coef=0.5,
    max_grad_norm=0.5,     
    verbose=1,
    tensorboard_log="./ppo_car_tensorboard/",
    device="auto",         
)

e:\Python\Python311\Lib\site-packages\stable_baselines3\common\utils.py:195: UserWarning: get_linear_fn() is deprecated, please use LinearSchedule() instead
  warnings.warn("get_linear_fn() is deprecated, please use LinearSchedule() instead")


Using cuda device


## 5. Train Model

Uses `EvalCallback` to automatically save the best model during training, and `CheckpointCallback` for periodic saves.

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)


eval_env_for_callback = build_env(n_envs=1)

# Save best model automatically during training
eval_callback = EvalCallback(
    eval_env_for_callback,
    best_model_save_path="./models/best/",
    log_path="./logs/eval/",
    eval_freq=10_000,       
    n_eval_episodes=5,
    deterministic=True,
    render=False,
)

# Periodic checkpoints every 50k steps
checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path="./models/checkpoints/",
    name_prefix="ppo_car",
)


model.learn(
    total_timesteps=1_000_000,
    callback=[eval_callback, checkpoint_callback],
    reset_num_timesteps=True,
)
model.save("models/ppo_car_racing")

Logging to ./ppo_car_tensorboard/PPO_1
----------------------------
| time/              |     |
|    fps             | 39  |
|    iterations      | 1   |
|    time_elapsed    | 13  |
|    total_timesteps | 512 |
----------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1e+03        |
|    ep_rew_mean          | -56.7        |
| time/                   |              |
|    fps                  | 37           |
|    iterations           | 2            |
|    time_elapsed         | 27           |
|    total_timesteps      | 1024         |
| train/                  |              |
|    approx_kl            | 0.0042809825 |
|    clip_fraction        | 0.019        |
|    clip_range           | 0.2          |
|    entropy_loss         | -4.26        |
|    explained_variance   | 0.00146      |
|    learning_rate        | 0.00025      |
|    loss                 | 0.318        |
|    n_updates            |

## 6. Continue Training from Checkpoint



In [ ]:
# Rebuild env 
train_env = build_env(n_envs=1)


model = PPO.load(
    "models/ppo_car_racing",
    env=train_env,
    custom_objects={
        "learning_rate": linear_schedule(2.5e-4),
        "clip_range": linear_schedule(0.2),
    }
)

eval_env_for_callback = build_env(n_envs=1)
eval_callback = EvalCallback(
    eval_env_for_callback,
    best_model_save_path="./models/best/",
    log_path="./logs/eval/",
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=True,
    render=False,
)

# Continue training
model.learn(
    total_timesteps=20,       
    callback=eval_callback,
    reset_num_timesteps=False,     
)

model.save("models/ppo_car_racing_v2")

Logging to ./ppo_car_tensorboard/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 978      |
|    ep_rew_mean     | 823      |
| time/              |          |
|    fps             | 77       |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 1000960  |
---------------------------------


## 7. Evaluate




In [ ]:

eval_env = build_env(n_envs=1)

episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,           
    n_eval_episodes=10,
    deterministic=True,
    return_episode_rewards=True,
)

eval_env.close()

mean_reward  = np.mean(episode_rewards)
std_dev      = np.std(episode_rewards)
variance     = np.var(episode_rewards)

ci_95        = 1.96 * std_dev / np.sqrt(len(episode_rewards))

print(f"Episode Rewards : {[round(r, 2) for r in episode_rewards]}")
print(f"Mean Reward     : {mean_reward:.2f}")
print(f"Std Deviation   : {std_dev:.2f}")
print(f"Variance        : {variance:.2f}")
print(f"95% CI          : {mean_reward:.2f} ± {ci_95:.2f}")
print(f"Avg Ep Length   : {np.mean(episode_lengths):.1f} steps")

Episode Rewards : [879.02, 538.71, 846.84, 708.14, 785.09, 856.23, 862.96, 805.84, 827.39, 900.0]
Mean Reward     : 801.02
Std Deviation   : 101.62
Variance        : 10327.19
95% CI          : 801.02 ± 62.99
Avg Ep Length   : 1000.0 steps


## 8. Visual Test (Human Render)



In [12]:

test_env = build_env(n_envs=1, render_mode="human")

obs = test_env.reset()

episodes = 5
ep = 0
ep_reward = 0.0

while ep < episodes:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    ep_reward += reward[0]  

    if done[0]:  
        ep += 1
        print(f"Episode {ep:2d} finished | Total Reward: {ep_reward:.2f}")
        ep_reward = 0.0
        obs = test_env.reset()

test_env.close()

Episode  1 finished | Total Reward: 836.88
Episode  2 finished | Total Reward: 822.08
Episode  3 finished | Total Reward: 901.20
Episode  4 finished | Total Reward: 821.05
Episode  5 finished | Total Reward: 811.48
